In [1]:
# %% Cell 0
import os

# Evita oversubscription: vários processos loky * várias threads BLAS/OpenMP.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")

import sys
import time
import numpy as np
import pandas as pd
import multiprocessing
import gc
import shutil
import tempfile
import uuid
from pathlib import Path
import joblib
from joblib import Parallel, delayed
from scipy.stats import norm
from tqdm import tqdm

# ML e Métricas
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, recall_score, f1_score, roc_auc_score, 
                             precision_score, matthews_corrcoef, precision_recall_curve, 
                             auc, average_precision_score)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
from cleverhans.tf2.attacks.carlini_wagner_l2 import carlini_wagner_l2 # >>> NOVO: Import do C&W <<<

# Modelos Customizados
if 'libs' not in sys.path: sys.path.append('libs')
import importlib
import libs.bloom_filter
import libs.wisard
importlib.reload(libs.bloom_filter)
importlib.reload(libs.wisard)
from libs.wisard import WiSARD

# ==========================================
# PAINEL DE CONTROLE DO EXPERIMENTO
# ==========================================
# DATASETS_TO_RUN = ['Bot-IoT', 'UNSW-NB15', 'CICIDS']
DATASETS_TO_RUN = ['CICIDS']
RUN_BINARY = True         
RUN_MULTICLASS = True      

ENCODING_TYPES = ['linear', 'gaussian', 'distributive']

# >>> NOVO: Adicionado 'C&W' à lista de ataques <<<
# ATTACKS_TO_RUN = ['FGSM', 'RANDOM_LINF', 'RANDOM_L2', 'C&W'] 
ATTACKS_TO_RUN = ['C&W'] 
EPSILON_LINF = 0.3   
EPSILON_L2 = 3.0     

# ==========================================
# CONTROLE DO C&W L2
# ==========================================
# Para teste rápido: reduza CW_BINARY_SEARCH_STEPS para 3 e CW_MAX_ITERATIONS para 100/200.
# Para execução final mais forte: 5 x 500 é mais caro, mas mais robusto.
CW_BATCH_SIZE = 128
CW_BINARY_SEARCH_STEPS = 5
CW_MAX_ITERATIONS = 500
CW_CONFIDENCE = 0.0
CW_LEARNING_RATE = 5e-3
CW_INITIAL_CONST = 1e-2
CW_ABORT_EARLY = True

N_JOBS = -1

# ==========================================
# CONTROLE DE PARALELISMO/MEMÓRIA - BLOOM WiSARD
# ==========================================
# A WiSARD é treinada uma única vez por processo worker e reutilizada em múltiplos chunks.
# Quanto maior BLOOM_CHUNKS_PER_WORKER, mais tarefas por worker e melhor balanceamento de carga,
# sem reenviar modelo por pickle. Use 2-8; 4 costuma ser um bom ponto inicial.
BLOOM_CHUNKS_PER_WORKER = 4

# None tenta usar /dev/shm no Linux, quando disponível, para memmap em RAM; caso contrário usa temp do SO.
BLOOM_MEMMAP_ROOT = None

# Candidatos para memmap em disco grande. O código escolhe automaticamente o primeiro com espaço.
BLOOM_MEMMAP_ROOT_CANDIDATES = [
    r"D:\wisard_memmap",
    r"E:\wisard_memmap",
    r"F:\wisard_memmap",
    r"C:\wisard_memmap",
    str(Path.cwd() / "wisard_memmap"),
]

# Margens de segurança para memmap/joblib.
BLOOM_MIN_FREE_DISK_GB = 80
BLOOM_DISK_SAFETY_MULTIPLIER = 1.35
BLOOM_CLEAN_OLD_MEMMAP_DIRS = True

# Controle de memória para escolher n_jobs seguro.
# A Bloom WiSARD já é mais leve que a Standard WiSARD com .tolist(), mas cada worker
# ainda treina sua própria cópia do modelo. Por isso, usamos estimativa dinâmica.
BLOOM_MEMORY_SAFETY_FRACTION = 0.65
BLOOM_RESERVED_MEMORY_GB = 32
BLOOM_MODEL_OVERHEAD_GB = 2


# True força encerramento dos workers ao fim de cada combinação, liberando os modelos grandes da RAM.
BLOOM_FORCE_WORKER_SHUTDOWN = True

# Semente para manter o mapeamento interno da WiSARD consistente entre workers.
WISARD_RANDOM_SEED = 42

os.makedirs('relatorios final', exist_ok=True)
os.makedirs('curves_data', exist_ok=True) 

# ==========================================
# SAÍDA DAS CURVAS EM DISCO GRANDE
# ==========================================
# Os CSVs continuam em "relatorios final". Apenas os .npz de curves_data podem ir para D:/E:/F:
# para evitar lotar o C:.
CURVES_OUTPUT_ROOT = None
CURVES_OUTPUT_ROOT_CANDIDATES = [
    r"D:\wisard_outputs\curves_data",
    r"E:\wisard_outputs\curves_data",
    r"F:\wisard_outputs\curves_data",
    str(Path.cwd() / "curves_data"),
]
CURVES_MIN_FREE_GB = 20
CURVES_SAVE_COMPRESSED = True

def _choose_curves_dir():
    candidates = []
    if CURVES_OUTPUT_ROOT is not None:
        candidates.append(CURVES_OUTPUT_ROOT)
    candidates.extend(CURVES_OUTPUT_ROOT_CANDIDATES)

    checked = []
    for candidate in candidates:
        try:
            p = Path(candidate).expanduser().resolve()
            parent = p
            while not parent.exists() and parent.parent != parent:
                parent = parent.parent
            if not parent.exists():
                checked.append((candidate, "parent_not_found", 0))
                continue
            free = shutil.disk_usage(parent).free
            checked.append((candidate, "ok", free))
            if free >= CURVES_MIN_FREE_GB * (1024 ** 3):
                p.mkdir(parents=True, exist_ok=True)
                return p
        except Exception as e:
            checked.append((candidate, f"error={e}", 0))

    checked_msg = "\n".join(
        f" - {path} | status={status} | livre={free / (1024**3):.1f} GB"
        for path, status, free in checked
    )
    raise OSError(
        "Nenhum diretório para curves_data tem espaço suficiente.\n"
        f"Espaço mínimo exigido: {CURVES_MIN_FREE_GB} GB\n"
        f"Pastas verificadas:\n{checked_msg}"
    )

CURVES_DIR = _choose_curves_dir()

def save_curve_npz(file_path, **kwargs):
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = file_path.with_suffix(file_path.suffix + ".tmp")
    try:
        with open(tmp_path, "wb") as f:
            if CURVES_SAVE_COMPRESSED:
                np.savez_compressed(f, **kwargs)
            else:
                np.savez(f, **kwargs)
        os.replace(tmp_path, file_path)
        return True
    except Exception:
        try:
            if tmp_path.exists():
                tmp_path.unlink()
        except Exception:
            pass
        raise


print(f"Datasets Selecionados: {DATASETS_TO_RUN}")
print(f"Modos Ativados: Binário={RUN_BINARY} | Multiclasse={RUN_MULTICLASS}")
print(f"Ataques Ativados: {ATTACKS_TO_RUN}")
print(f"Binarizações: {ENCODING_TYPES}")
print(f"Diretório das curvas (.npz): {CURVES_DIR}")



Datasets Selecionados: ['CICIDS']
Modos Ativados: Binário=True | Multiclasse=True
Ataques Ativados: ['C&W']
Binarizações: ['linear', 'gaussian', 'distributive']
Diretório das curvas (.npz): D:\wisard_outputs\curves_data


In [2]:
# %% Cell 1
# ==========================================
# FUNÇÕES DE MODELAÇÃO, PARALELISMO E MÉTRICAS
# ==========================================
def build_and_train_mlp(X, y_cat, num_classes):
    inputs = Input(shape=(X.shape[1],))
    x = Dense(256, activation='relu')(inputs)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.4)(x)
    logits = Dense(num_classes, name='logits')(x)
    outputs = Activation('softmax')(logits)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer=Adam(0.001), metrics=['accuracy'])
    model.fit(X, y_cat, batch_size=64, epochs=5, verbose=0)
    return model

def process_data_vectorized_sequential(data, resolution, enc_type, custom_thresholds=None, chunk_size=20000):
    n_samples, n_features = data.shape
    result = np.empty((n_samples, n_features * resolution), dtype=np.int8)
    
    if enc_type == 'linear':
        indices = np.arange(resolution, dtype=np.int8)
        
    total_chunks = (n_samples + chunk_size - 1) // chunk_size
    
    for i in range(total_chunks):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, n_samples)
        chunk = data[start:end]
        
        if enc_type in ['gaussian', 'distributive']:
            bits = (chunk[:, :, None] >= custom_thresholds[None, :, :]).astype(np.int8)
        elif enc_type == 'linear':
            chunk_clipped = np.clip(chunk, 0.0, 1.0)
            limits = (chunk_clipped * resolution).astype(np.int8)
            bits = (limits[:, :, None] > indices[None, None, :]).astype(np.int8)
            
        result[start:end] = bits.reshape(chunk.shape[0], -1)
    return result


# --------------------------------------------------------------------------
# MÓDULO IMPORTÁVEL PARA OS WORKERS LOKY
# --------------------------------------------------------------------------
# Motivo: funções definidas apenas no notebook podem ser serializadas por cloudpickle.
# Este módulo real permite manter estado global por processo: cada worker treina a Bloom
# WiSARD uma única vez e reutiliza o modelo nos chunks seguintes.
BLOOM_WORKER_MODULE_CODE = r"""
import os

# Evita oversubscription dentro de cada worker.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")

import sys
import time
import random
import gc
import numpy as np
import joblib

if 'libs' not in sys.path:
    sys.path.append('libs')
from libs.wisard import WiSARD

_STATE_KEY = None
_MODEL = None
_X_TEST = None
_ADV = None
_PAD_BUFFER = None
_TRAIN_TIME = 0.0


def _ensure_state(config):
    global _STATE_KEY, _MODEL, _X_TEST, _ADV, _PAD_BUFFER, _TRAIN_TIME

    if _STATE_KEY == config['state_key'] and _MODEL is not None:
        return 0.0

    np.random.seed(int(config.get('seed', 42)))
    random.seed(int(config.get('seed', 42)))

    X_train = joblib.load(config['X_train_path'], mmap_mode='r')
    y_train = joblib.load(config['y_train_path'], mmap_mode='r')
    _X_TEST = joblib.load(config['X_test_path'], mmap_mode='r')
    _ADV = {name: joblib.load(path, mmap_mode='r') for name, path in config['adv_paths'].items()}

    t0 = time.perf_counter()
    model = WiSARD(
        int(config['num_inputs']),
        int(config['num_classes']),
        int(config['addr']),
        int(config['bloom']),
        int(config['h'])
    )

    # Treino sem .tolist(): cada worker percorre o memmap read-only.
    for i in range(len(X_train)):
        model.train(X_train[i], int(y_train[i]))

    _TRAIN_TIME = time.perf_counter() - t0
    _MODEL = model
    _STATE_KEY = config['state_key']

    # Buffer reutilizável por processo para evitar np.pad por amostra.
    padded_len = int(config['num_inputs']) + int(getattr(model, 'pad_zeros', 0))
    _PAD_BUFFER = np.zeros(padded_len, dtype=X_train.dtype)

    del X_train, y_train
    gc.collect()

    return _TRAIN_TIME


def _predict_dynamic_with_prob(chunk):
    global _MODEL, _PAD_BUFFER

    model = _MODEL
    pad_zeros = int(model.pad_zeros)
    input_order = model.input_order
    discriminators = model.discriminators
    num_classes = len(discriminators)

    n_samples = len(chunk)
    preds = np.empty(n_samples, dtype=np.int32)
    probs = np.empty((n_samples, num_classes), dtype=np.float32)

    for row_idx, sample in enumerate(chunk):
        if pad_zeros > 0:
            _PAD_BUFFER[:len(sample)] = sample
            _PAD_BUFFER[len(sample):] = 0
            xv_padded = _PAD_BUFFER[input_order]
        else:
            xv_padded = sample[input_order]

        b = 1
        last_tie = np.array([0], dtype=np.int32)
        first_responses = None

        # Bleaching exponencial: evita b=1,2,3,... quando há empates muito longos.
        while b <= 200:
            model.set_bleaching(b)

            responses = np.empty(num_classes, dtype=np.float32)
            for c_idx, disc in enumerate(discriminators):
                responses[c_idx] = disc.predict(xv_padded)

            if b == 1:
                first_responses = responses.copy()

            max_res = responses.max()
            if max_res == 0:
                break

            winners = np.flatnonzero(responses == max_res)
            last_tie = winners
            if len(winners) == 1:
                break

            b += max(1, int(b * 0.2))

        pred_class = int(last_tie[0])
        preds[row_idx] = pred_class

        if first_responses is not None:
            total = float(first_responses.sum())
            if total > 0:
                probs[row_idx] = first_responses / total
            else:
                probs[row_idx] = 0.0
                probs[row_idx, pred_class] = 1.0
        else:
            probs[row_idx] = 0.0
            probs[row_idx, pred_class] = 1.0

    return preds, probs


def evaluate_bloom_range_worker(config, start, end):
    train_time = _ensure_state(config)

    start = int(start)
    end = int(end)
    pid = os.getpid()

    t1 = time.perf_counter()
    yp_clean, yp_clean_prob = _predict_dynamic_with_prob(_X_TEST[start:end])
    infer_time_clean = time.perf_counter() - t1

    yp_adv = {}
    yp_adv_prob = {}
    infer_time_adv = {}

    for atk_name, adv_matrix in _ADV.items():
        t2 = time.perf_counter()
        p, pr = _predict_dynamic_with_prob(adv_matrix[start:end])
        yp_adv[atk_name] = p
        yp_adv_prob[atk_name] = pr
        infer_time_adv[atk_name] = time.perf_counter() - t2

    return {
        'pid': pid,
        'start': start,
        'end': end,
        'yp_clean': yp_clean,
        'yp_clean_prob': yp_clean_prob,
        'yp_adv': yp_adv,
        'yp_adv_prob': yp_adv_prob,
        'train_time': float(train_time),
        'infer_time_clean': float(infer_time_clean),
        'infer_time_adv': infer_time_adv,
    }
"""


def _write_bloom_worker_runtime_module(module_path='bloom_worker_runtime.py'):
    module_path = Path(module_path)
    module_path.write_text(BLOOM_WORKER_MODULE_CODE, encoding='utf-8')
    import importlib
    importlib.invalidate_caches()
    if str(module_path.parent.resolve()) not in sys.path:
        sys.path.insert(0, str(module_path.parent.resolve()))
    return module_path


def _get_available_memory_bytes():
    try:
        import psutil
        return int(psutil.virtual_memory().available)
    except Exception:
        pass

    if os.name == "nt":
        try:
            import ctypes

            class MEMORYSTATUSEX(ctypes.Structure):
                _fields_ = [
                    ("dwLength", ctypes.c_ulong),
                    ("dwMemoryLoad", ctypes.c_ulong),
                    ("ullTotalPhys", ctypes.c_ulonglong),
                    ("ullAvailPhys", ctypes.c_ulonglong),
                    ("ullTotalPageFile", ctypes.c_ulonglong),
                    ("ullAvailPageFile", ctypes.c_ulonglong),
                    ("ullTotalVirtual", ctypes.c_ulonglong),
                    ("ullAvailVirtual", ctypes.c_ulonglong),
                    ("sullAvailExtendedVirtual", ctypes.c_ulonglong),
                ]

            stat = MEMORYSTATUSEX()
            stat.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
            ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(stat))
            return int(stat.ullAvailPhys)
        except Exception:
            return None

    try:
        pages = os.sysconf("SC_AVPHYS_PAGES")
        page_size = os.sysconf("SC_PAGE_SIZE")
        return int(pages * page_size)
    except Exception:
        return None


def _existing_parent(path):
    p = Path(path).expanduser().resolve()
    while not p.exists() and p.parent != p:
        p = p.parent
    return p if p.exists() else None


def _cleanup_bloom_memmap_leftovers(root):
    if not BLOOM_CLEAN_OLD_MEMMAP_DIRS:
        return
    root = Path(root)
    if not root.exists():
        return
    for p in root.glob("bloom_wisard_memmap_*"):
        try:
            shutil.rmtree(p, ignore_errors=True)
        except Exception:
            pass


def _estimate_memmap_required_bytes(X_train_bin, y_train_curr, X_test_bin, active_adv_dict):
    total = 0
    total += int(np.ascontiguousarray(X_train_bin).nbytes)
    total += int(np.asarray(y_train_curr, dtype=np.int32).nbytes)
    total += int(np.ascontiguousarray(X_test_bin).nbytes)
    for X_adv in active_adv_dict.values():
        total += int(np.ascontiguousarray(X_adv).nbytes)

    total = int(total * float(BLOOM_DISK_SAFETY_MULTIPLIER))
    total += int(BLOOM_MIN_FREE_DISK_GB * (1024 ** 3))
    return total


def _get_memmap_root(required_bytes=0):
    candidates = []
    if BLOOM_MEMMAP_ROOT is not None:
        candidates.append(BLOOM_MEMMAP_ROOT)
    candidates.extend(BLOOM_MEMMAP_ROOT_CANDIDATES)
    candidates.append(tempfile.gettempdir())

    checked = []
    for candidate in candidates:
        try:
            parent = _existing_parent(candidate)
            if parent is None:
                checked.append((candidate, "parent_not_found", 0))
                continue

            free = int(shutil.disk_usage(parent).free)
            checked.append((candidate, "ok", free))

            if free >= int(required_bytes):
                os.makedirs(candidate, exist_ok=True)
                _cleanup_bloom_memmap_leftovers(candidate)
                return str(Path(candidate).resolve())
        except Exception as e:
            checked.append((candidate, f"error={e}", 0))

    checked_msg = "\n".join(
        f" - {path} | status={status} | livre={free / (1024**3):.1f} GB"
        for path, status, free in checked
    )

    raise OSError(
        "Nenhum diretório de memmap tem espaço livre suficiente.\n"
        f"Espaço estimado necessário, com margem: {required_bytes / (1024**3):.1f} GB\n"
        "Pastas verificadas:\n"
        f"{checked_msg}\n\n"
        "Solução prática: libere espaço em disco ou defina BLOOM_MEMMAP_ROOT "
        "para uma unidade grande, por exemplo r'D:\\wisard_memmap'."
    )


def _estimate_bloom_worker_peak_bytes(num_inputs, num_classes, addr, bloom, X_test_bin, active_adv_dict, n_jobs_guess):
    """
    Estimativa conservadora do pico de RAM por worker da Bloom WiSARD.

    Cada worker treina uma cópia completa do modelo. O maior bloco esperado é:
        num_classes * n_rams * bloom
    somado a buffers temporários e arrays de probabilidade por chunk.
    """
    num_inputs = int(num_inputs)
    num_classes = int(num_classes)
    addr = int(addr)
    bloom = int(bloom)

    n_rams = int(np.ceil(num_inputs / max(1, addr)))

    # Modelo: assume contadores/estruturas internas com margem conservadora.
    model_bytes = n_rams * num_classes * bloom * 8

    n_test = int(X_test_bin.shape[0])
    chunks_per_worker = int(BLOOM_CHUNKS_PER_WORKER)
    approx_tasks = max(1, int(n_jobs_guess) * chunks_per_worker)
    approx_chunk_rows = max(1, int(np.ceil(n_test / approx_tasks)))

    # Retornos de probabilidade/predição por chunk clean + um ataque por vez.
    chunk_result_bytes = approx_chunk_rows * num_classes * 4 * 2
    chunk_input_bytes = approx_chunk_rows * int(X_test_bin.shape[1])

    overhead = int(BLOOM_MODEL_OVERHEAD_GB * (1024 ** 3))

    return int(model_bytes + chunk_result_bytes + chunk_input_bytes + overhead)


def _resolve_cpu_jobs(n_jobs, n_samples):
    cpu_count = multiprocessing.cpu_count()
    if n_jobs is None or n_jobs == -1:
        resolved = max(1, cpu_count - 1)
    elif n_jobs < 0:
        resolved = max(1, cpu_count + 1 + int(n_jobs))
    else:
        resolved = int(n_jobs)
    return max(1, min(resolved, int(n_samples)))


def _resolve_n_jobs_memory_safe(n_jobs, n_samples, num_inputs, num_classes, addr, bloom, X_test_bin, active_adv_dict):
    cpu_jobs = _resolve_cpu_jobs(n_jobs, n_samples)
    available = _get_available_memory_bytes()

    if available is None:
        safe_jobs = min(cpu_jobs, 4)
        print(
            f"     [MEM] Não consegui medir RAM disponível. "
            f"Usando fallback seguro: n_jobs={safe_jobs}/{cpu_jobs}"
        )
        return max(1, safe_jobs)

    reserved = int(BLOOM_RESERVED_MEMORY_GB * (1024 ** 3))
    usable = int(max(1, available * BLOOM_MEMORY_SAFETY_FRACTION - reserved))

    per_worker = _estimate_bloom_worker_peak_bytes(
        num_inputs=num_inputs,
        num_classes=num_classes,
        addr=addr,
        bloom=bloom,
        X_test_bin=X_test_bin,
        active_adv_dict=active_adv_dict,
        n_jobs_guess=cpu_jobs
    )

    max_jobs_by_memory = max(1, int(usable // max(1, per_worker)))
    resolved = max(1, min(cpu_jobs, max_jobs_by_memory, int(n_samples)))

    print(
        "     [MEM] RAM disponível≈"
        f"{available / (1024**3):.1f} GB | "
        f"RAM utilizável≈{usable / (1024**3):.1f} GB | "
        f"pico estimado/worker≈{per_worker / (1024**3):.1f} GB | "
        f"n_jobs seguro={resolved}/{cpu_jobs}"
    )

    return resolved


def _build_ranges(n_samples, n_jobs, chunks_per_worker=4):
    n_tasks = max(1, min(int(n_samples), int(n_jobs) * int(chunks_per_worker)))
    cuts = np.linspace(0, int(n_samples), n_tasks + 1, dtype=np.int64)
    return [(int(cuts[i]), int(cuts[i + 1])) for i in range(n_tasks) if cuts[i] < cuts[i + 1]]


def _dump_memmap_array(array, folder, name):
    path = os.path.join(folder, f'{name}.joblib')
    arr = np.ascontiguousarray(array)

    required = int(arr.nbytes * 1.10) + int(2 * (1024 ** 3))
    free = shutil.disk_usage(folder).free

    if free < required:
        raise OSError(
            f"Espaço insuficiente ao gravar {name}.joblib.\n"
            f"Necessário≈{required / (1024**3):.1f} GB | "
            f"Livre≈{free / (1024**3):.1f} GB | "
            f"Pasta={folder}"
        )

    joblib.dump(arr, path, compress=0)
    return path


def _safe_rmtree(path, retries=8, sleep_s=1.0):
    path = Path(path)
    if not path.exists():
        return True

    for attempt in range(1, retries + 1):
        try:
            gc.collect()
            shutil.rmtree(path, ignore_errors=False)
            return True
        except Exception as e:
            if attempt == retries:
                print(
                    f"     [WARN] Não consegui apagar temp_dir após {retries} tentativas: {path}\n"
                    f"            Motivo: {repr(e)}\n"
                    f"            Você pode apagar manualmente depois."
                )
                return False
            time.sleep(sleep_s)

    return False


def _shutdown_loky_workers():
    try:
        from joblib.externals.loky import get_reusable_executor
        get_reusable_executor().shutdown(wait=True, kill_workers=True)
    except Exception:
        pass


def evaluate_bloom_persistent_parallel(
    num_inputs,
    num_classes,
    addr,
    bloom,
    h,
    X_train_bin,
    y_train_curr,
    X_test_bin,
    active_adv_dict,
    n_jobs=-1,
    chunks_per_worker=None,
):
    """
    Avaliação paralela persistente da Bloom WiSARD.

    Mantém a arquitetura loky + memmap + worker isolado, com:
    1. escolha automática de disco para memmap;
    2. checagem de espaço antes do joblib.dump;
    3. n_jobs máximo seguro por RAM;
    4. temp_folder explícito para o joblib;
    5. encerramento dos workers ao fim de cada combinação.
    """
    if chunks_per_worker is None:
        chunks_per_worker = BLOOM_CHUNKS_PER_WORKER

    n_samples = len(X_test_bin)

    required_disk = _estimate_memmap_required_bytes(
        X_train_bin=X_train_bin,
        y_train_curr=y_train_curr,
        X_test_bin=X_test_bin,
        active_adv_dict=active_adv_dict
    )

    memmap_root = _get_memmap_root(required_bytes=required_disk)
    os.makedirs(memmap_root, exist_ok=True)

    print(
        f"     [DISK] Memmap root: {memmap_root} | "
        f"livre≈{shutil.disk_usage(memmap_root).free / (1024**3):.1f} GB | "
        f"estimado necessário≈{required_disk / (1024**3):.1f} GB"
    )

    n_jobs_resolved = _resolve_n_jobs_memory_safe(
        n_jobs=n_jobs,
        n_samples=n_samples,
        num_inputs=num_inputs,
        num_classes=num_classes,
        addr=addr,
        bloom=bloom,
        X_test_bin=X_test_bin,
        active_adv_dict=active_adv_dict
    )

    ranges = _build_ranges(n_samples, n_jobs_resolved, chunks_per_worker)

    _write_bloom_worker_runtime_module()
    import importlib
    runtime = importlib.import_module('bloom_worker_runtime')
    runtime = importlib.reload(runtime)

    temp_dir = tempfile.mkdtemp(prefix='bloom_wisard_memmap_', dir=memmap_root)
    t_wall = time.perf_counter()

    try:
        X_train_path = _dump_memmap_array(X_train_bin, temp_dir, 'X_train_bin')
        y_train_path = _dump_memmap_array(np.asarray(y_train_curr, dtype=np.int32), temp_dir, 'y_train_curr')
        X_test_path = _dump_memmap_array(X_test_bin, temp_dir, 'X_test_bin')
        adv_paths = {
            atk_name: _dump_memmap_array(X_adv, temp_dir, f'adv_{atk_name}')
            for atk_name, X_adv in active_adv_dict.items()
        }

        config = {
            'state_key': str(uuid.uuid4()),
            'seed': int(WISARD_RANDOM_SEED),
            'num_inputs': int(num_inputs),
            'num_classes': int(num_classes),
            'addr': int(addr),
            'bloom': int(bloom),
            'h': int(h),
            'X_train_path': X_train_path,
            'y_train_path': y_train_path,
            'X_test_path': X_test_path,
            'adv_paths': adv_paths,
        }

        results = Parallel(
            n_jobs=n_jobs_resolved,
            backend='loky',
            batch_size=1,
            pre_dispatch=n_jobs_resolved,
            max_nbytes=None,
            temp_folder=temp_dir,
            verbose=0,
        )(
            delayed(runtime.evaluate_bloom_range_worker)(config, start, end)
            for start, end in ranges
        )

        wall_time = time.perf_counter() - t_wall

    finally:
        if BLOOM_FORCE_WORKER_SHUTDOWN:
            _shutdown_loky_workers()
        gc.collect()
        _safe_rmtree(temp_dir)

    return {
        'results': results,
        'n_jobs': n_jobs_resolved,
        'ranges': ranges,
        'parallel_wall_time': wall_time,
    }

def calculate_miss_rates(y_true, y_pred, y_prob, context_name, class_names):
    metrics = {}
    try:
        normal_idx = next(i for i, name in enumerate(class_names) if 'normal' in str(name).lower())
    except StopIteration:
        normal_idx = 0
        
    is_binary = len(class_names) == 2
    avg_type = 'binary' if is_binary else 'weighted'
    pos_label = 1 if is_binary else None
    
    metrics[f'{context_name}_Acc'] = accuracy_score(y_true, y_pred)
    metrics[f'{context_name}_Precision'] = precision_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_Recall'] = recall_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_F1'] = f1_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_MCC'] = matthews_corrcoef(y_true, y_pred)
    
    mask_normal = (y_true == normal_idx)
    mask_attack = (y_true != normal_idx)
    
    if np.sum(mask_normal) > 0:
        metrics[f'{context_name}_FAR'] = np.sum((y_pred != normal_idx) & mask_normal) / np.sum(mask_normal)
    else:
        metrics[f'{context_name}_FAR'] = 0.0

    if np.sum(mask_attack) > 0:
        metrics[f'{context_name}_ASR'] = np.sum((y_pred == normal_idx) & mask_attack) / np.sum(mask_attack)
    else:
        metrics[f'{context_name}_ASR'] = 0.0

    try:
        if is_binary:
            prob_positive = y_prob[:, 1] if len(y_prob.shape) > 1 else y_prob
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, prob_positive)
            prec, rec, _ = precision_recall_curve(y_true, prob_positive)
            metrics[f'{context_name}_PR_AUC'] = auc(rec, prec)
        else:
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, y_prob, multi_class='ovr')
            y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
            metrics[f'{context_name}_PR_AUC'] = average_precision_score(y_true_bin, y_prob, average="macro")
    except Exception as e:
        metrics[f'{context_name}_AUC'] = 0.0
        metrics[f'{context_name}_PR_AUC'] = 0.0
    
    if not is_binary:
        metrics[f'{context_name}_F1_Macro'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    for idx, name in enumerate(class_names):
        if idx == normal_idx: continue
        mask_t = (y_true == idx)
        if np.sum(mask_t) > 0:
            metrics[f'{context_name}_Miss_{name}'] = np.sum((y_pred == normal_idx) & mask_t) / np.sum(mask_t)
        else:
            metrics[f'{context_name}_Miss_{name}'] = 0.0
            
    return metrics




In [ ]:
# %% Cell 2
# ==========================================
# LOOP PRINCIPAL DO EXPERIMENTO
# ==========================================
for dataset_name in DATASETS_TO_RUN:
    print(f"\n{'='*50}")
    print(f">>> A INICIAR EXPERIMENTOS: {dataset_name.upper()}")
    print(f"{'='*50}")
            
    # 1. CARREGAMENTO E PRÉ-PROCESSAMENTO
    if dataset_name == 'Bot-IoT':
        df_train = pd.read_csv("data2/BotIoT_training-set.csv")
        df_test = pd.read_csv("data2/BotIoT_testing-set.csv")
    elif dataset_name == 'UNSW-NB15':
        df_train = pd.read_csv("data/UNSW_NB15_training-set.csv")
        df_test = pd.read_csv("data/UNSW_NB15_testing-set.csv")
    elif dataset_name == 'CICIDS':
        df_train = pd.read_csv("data3/CICIDS_training-set.csv")
        df_test = pd.read_csv("data3/CICIDS_testing-set.csv")
        
    for df in [df_train, df_test]:
        if 'id' in df.columns: df.drop(columns=['id'], inplace=True)

    y_train_bin = df_train['label'].values
    y_test_bin = df_test['label'].values
    class_names_bin = ['Normal', 'Attack']

    df_train['attack_cat'] = df_train['attack_cat'].astype(str).str.strip().str.lower()
    df_test['attack_cat'] = df_test['attack_cat'].astype(str).str.strip().str.lower()
    le = LabelEncoder()
    le.fit(pd.concat([df_train['attack_cat'], df_test['attack_cat']]))
    y_train_multi = le.transform(df_train['attack_cat'])
    y_test_multi = le.transform(df_test['attack_cat'])
    class_names_multi = le.classes_

    df_train.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')
    df_test.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')

    categorical_cols = df_train.select_dtypes(include=['object']).columns
    numerical_cols = df_train.select_dtypes(include=['int64', 'float64']).columns

    preprocessor = ColumnTransformer([
        ('num', MinMaxScaler(feature_range=(0,1)), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

    print(">>> A aplicar Scaling e One-Hot Encoding...")
    preprocessor.fit(df_train)
    X_train = preprocessor.transform(df_train).astype('float32')
    X_test = preprocessor.transform(df_test).astype('float32')

    del df_train, df_test
    gc.collect()

    # 2. GERAÇÃO DOS ATAQUES ADVERSARIAIS E RUÍDOS
    attacks_dict_bin = {}
    attacks_dict_multi = {}

    if 'FGSM' in ATTACKS_TO_RUN:
        print(">>> A gerar Ataque FGSM (Caixa-Branca Transferida)...")
        if RUN_BINARY:
            mlp_bin = build_and_train_mlp(X_train, to_categorical(y_train_bin, 2), 2)
            logits_bin = Model(inputs=mlp_bin.input, outputs=mlp_bin.get_layer('logits').output)
            attacks_dict_bin['FGSM'] = fast_gradient_method(logits_bin, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
            del mlp_bin, logits_bin 
        
        if RUN_MULTICLASS:
            mlp_multi = build_and_train_mlp(X_train, to_categorical(y_train_multi, len(class_names_multi)), len(class_names_multi))
            logits_multi = Model(inputs=mlp_multi.input, outputs=mlp_multi.get_layer('logits').output)
            attacks_dict_multi['FGSM'] = fast_gradient_method(logits_multi, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
            del mlp_multi, logits_multi 
        gc.collect()

    if 'RANDOM_LINF' in ATTACKS_TO_RUN:
        print(f">>> A gerar Ruído Aleatório L_infinito (Eps={EPSILON_LINF})...")
        noise = np.random.uniform(-EPSILON_LINF, EPSILON_LINF, X_test.shape).astype('float32')
        if RUN_BINARY: attacks_dict_bin['RANDOM_LINF'] = np.clip(X_test + noise, 0.0, 1.0)
        if RUN_MULTICLASS: attacks_dict_multi['RANDOM_LINF'] = np.clip(X_test + noise, 0.0, 1.0)
        del noise

    if 'RANDOM_L2' in ATTACKS_TO_RUN:
        print(f">>> A gerar Ruído Aleatório L_2 (Eps={EPSILON_L2})...")
        noise = np.random.normal(0, 1, X_test.shape).astype('float32')
        norms = np.linalg.norm(noise, axis=1, keepdims=True)
        norms[norms == 0] = 1e-10
        noise = noise * (EPSILON_L2 / norms)
        if RUN_BINARY: attacks_dict_bin['RANDOM_L2'] = np.clip(X_test + noise, 0.0, 1.0)
        if RUN_MULTICLASS: attacks_dict_multi['RANDOM_L2'] = np.clip(X_test + noise, 0.0, 1.0)
        del noise

    # >>> BLOCO DO ATAQUE C&W L2 OTIMIZADO <<<
    if 'C&W' in ATTACKS_TO_RUN:
        print(">>> A gerar Ataque C&W L2 via surrogate MLP...")

        def generate_cw_in_batches(
            logits_model,
            X_np,
            attack_name="C&W",
            batch_size=CW_BATCH_SIZE,
            binary_search_steps=CW_BINARY_SEARCH_STEPS,
            max_iterations=CW_MAX_ITERATIONS,
            confidence=CW_CONFIDENCE,
            learning_rate=CW_LEARNING_RATE,
            initial_const=CW_INITIAL_CONST,
            abort_early=CW_ABORT_EARLY,
        ):
            """
            Gera C&W L2 em lotes com menor pico de RAM.

            Melhorias:
            1. prealoca adv_x em vez de acumular lista + np.vstack;
            2. mede sucesso no surrogate e norma L2;
            3. usa wrapper 4D->2D para compatibilidade com implementações C&W
               que assumem entrada tipo imagem no mascaramento interno;
            4. não altera as colunas do relatório final.
            """
            X_np = np.asarray(X_np, dtype=np.float32)
            n_samples = len(X_np)
            n_features = int(X_np.shape[1])

            adv_x = np.empty_like(X_np, dtype=np.float32)
            l2_norms = np.zeros(n_samples, dtype=np.float32)
            success_flags = np.zeros(n_samples, dtype=bool)

            total_batches = (n_samples + batch_size - 1) // batch_size

            def cw_model_fn(x4):
                # CleverHans C&W pode trabalhar melhor com entrada 4D.
                # O surrogate MLP, porém, espera entrada 2D.
                x2 = tf.reshape(x4, (tf.shape(x4)[0], n_features))
                return logits_model(x2, training=False)

            for i in tqdm(range(total_batches), desc=f"Gerando {attack_name}", unit="lote"):
                start_idx = i * batch_size
                end_idx = min((i + 1) * batch_size, n_samples)

                x_batch_np = X_np[start_idx:end_idx].astype(np.float32, copy=False)
                x_batch_np = np.clip(x_batch_np, 0.0, 1.0)

                # Predição limpa do surrogate no espaço 2D tabular.
                clean_logits = logits_model(
                    tf.convert_to_tensor(x_batch_np, dtype=tf.float32),
                    training=False
                )
                pred_clean = tf.argmax(clean_logits, axis=1).numpy()

                # C&W em formato 4D para evitar problemas internos de shape.
                x_batch_4d = x_batch_np.reshape((x_batch_np.shape[0], n_features, 1, 1))
                x_batch_tf = tf.convert_to_tensor(x_batch_4d, dtype=tf.float32)

                adv_batch_4d = carlini_wagner_l2(
                    cw_model_fn,
                    x_batch_tf,
                    batch_size=x_batch_tf.shape[0],
                    clip_min=0.0,
                    clip_max=1.0,
                    binary_search_steps=binary_search_steps,
                    max_iterations=max_iterations,
                    abort_early=abort_early,
                    confidence=confidence,
                    initial_const=initial_const,
                    learning_rate=learning_rate,
                )

                adv_batch_np = np.asarray(adv_batch_4d, dtype=np.float32).reshape((-1, n_features))
                adv_batch_np = np.clip(adv_batch_np, 0.0, 1.0)

                # Predição adversarial do surrogate no espaço 2D tabular.
                adv_logits = logits_model(
                    tf.convert_to_tensor(adv_batch_np, dtype=tf.float32),
                    training=False
                )
                pred_adv = tf.argmax(adv_logits, axis=1).numpy()

                delta = adv_batch_np - x_batch_np
                l2_batch = np.linalg.norm(delta.reshape(delta.shape[0], -1), axis=1)

                adv_x[start_idx:end_idx] = adv_batch_np
                l2_norms[start_idx:end_idx] = l2_batch
                success_flags[start_idx:end_idx] = (pred_adv != pred_clean)

                del x_batch_np, x_batch_4d, x_batch_tf, adv_batch_4d, adv_batch_np
                del clean_logits, adv_logits, pred_clean, pred_adv, delta, l2_batch
                gc.collect()

            print(
                f"   [{attack_name}] Sucesso no surrogate: {100*np.mean(success_flags):.2f}% | "
                f"L2 médio={np.mean(l2_norms):.4f} | "
                f"L2 mediano={np.median(l2_norms):.4f} | "
                f"L2 máx={np.max(l2_norms):.4f}"
            )

            return adv_x

        if RUN_BINARY:
            print("   -> Treinando surrogate MLP para C&W (Binário)...")
            mlp_bin = build_and_train_mlp(X_train, to_categorical(y_train_bin, 2), 2)
            logits_bin = Model(inputs=mlp_bin.input, outputs=mlp_bin.get_layer('logits').output)

            attacks_dict_bin['C&W'] = generate_cw_in_batches(
                logits_bin,
                X_test,
                attack_name="C&W Binário"
            )

            del mlp_bin, logits_bin
            gc.collect()
            tf.keras.backend.clear_session()

        if RUN_MULTICLASS:
            print("   -> Treinando surrogate MLP para C&W (Multiclasse)...")
            mlp_multi = build_and_train_mlp(
                X_train,
                to_categorical(y_train_multi, len(class_names_multi)),
                len(class_names_multi)
            )
            logits_multi = Model(inputs=mlp_multi.input, outputs=mlp_multi.get_layer('logits').output)

            attacks_dict_multi['C&W'] = generate_cw_in_batches(
                logits_multi,
                X_test,
                attack_name="C&W Multiclasse"
            )

            del mlp_multi, logits_multi
            gc.collect()
            tf.keras.backend.clear_session()

    gc.collect()

    # 3. HIPERPARÂMETROS
    param_grid = {
        'resolution': [1, 2, 4, 8, 10],
        'addressSize': [5, 10, 15, 20],
        'bloomSize': [1024, 2048],
        'hashes': [2, 3]
    }

    # param_grid = {
    #     'resolution': [10],
    #     'addressSize': [5, 10, 15, 20],
    #     'bloomSize': [1024, 2048],
    #     'hashes': [2, 3]
    # }


    # Pré-cálculo para termômetro gaussiano
    X_mean = X_train.mean(axis=0)
    X_std = X_train.std(axis=0)
    X_std[X_std == 0] = 1e-8 

    # 4. LOOP DA BLOOM WISARD
    for enc_type in ENCODING_TYPES:
        for res in param_grid['resolution']:

            print(f"\n[{dataset_name} | {enc_type.upper()} | Res={res}] Binarização C...")
            
            custom_thresh = None
            
            if enc_type == 'gaussian':
                skews = [norm.ppf((i+1)/(res+1)) for i in range(res)]
                custom_thresh = X_mean[:, None] + (X_std[:, None] * skews)
            
            elif enc_type == 'distributive':
                percentiles = np.linspace(0, 100, res + 2)[1:-1]
                custom_thresh = np.percentile(X_train, percentiles, axis=0).T 

            # Binarizando os dados limpos
            X_train_bin = process_data_vectorized_sequential(X_train, res, enc_type, custom_thresh)
            X_test_bin = process_data_vectorized_sequential(X_test, res, enc_type, custom_thresh)
            num_inputs = len(X_train_bin[0])
            
            # Binarizando matrizes de ataque ativas
            bin_adv_dict = {}
            if RUN_BINARY:
                for atk_name, X_adv_matrix in attacks_dict_bin.items():
                    bin_adv_dict[atk_name] = process_data_vectorized_sequential(X_adv_matrix, res, enc_type, custom_thresh)
                    
            multi_adv_dict = {}
            if RUN_MULTICLASS:
                for atk_name, X_adv_matrix in attacks_dict_multi.items():
                    multi_adv_dict[atk_name] = process_data_vectorized_sequential(X_adv_matrix, res, enc_type, custom_thresh)

            modes_to_run = []
            if RUN_BINARY: modes_to_run.append('binary')
            if RUN_MULTICLASS: modes_to_run.append('multiclass')

            for mode in modes_to_run:
                if mode == 'binary':
                    y_train_curr, y_test_curr, class_names_curr = y_train_bin, y_test_bin, class_names_bin
                    active_adv_dict = bin_adv_dict
                else:
                    y_train_curr, y_test_curr, class_names_curr = y_train_multi, y_test_multi, class_names_multi
                    active_adv_dict = multi_adv_dict

                num_classes = len(class_names_curr)
                csv_name = f'relatorios final/bloom_wisard_{dataset_name}_{mode}_{enc_type}.csv'

                for addr in param_grid['addressSize']:
                    for bloom in param_grid['bloomSize']:
                        for h in param_grid['hashes']:
                            print(f"     [Addr={addr} | Bloom={bloom} | Mode={mode} | Hash={h}] A treinar e avaliar WiSARD (True Multiprocessing)...")
                            
                            # 1/2. Paralelismo persistente com loky + memmap explícito
                            # Cada processo worker:
                            #   (a) carrega os arrays grandes via memmap read-only;
                            #   (b) instancia e treina a sua Bloom WiSARD uma única vez;
                            #   (c) processa um ou mais chunks de teste usando apenas start/end.
                            parallel_out = evaluate_bloom_persistent_parallel(
                                num_inputs=num_inputs,
                                num_classes=num_classes,
                                addr=addr,
                                bloom=bloom,
                                h=h,
                                X_train_bin=X_train_bin,
                                y_train_curr=y_train_curr,
                                X_test_bin=X_test_bin,
                                active_adv_dict=active_adv_dict,
                                n_jobs=N_JOBS
                            )
                            results = parallel_out['results']
                            results_sorted = sorted(results, key=lambda r: r['start'])
                            
                            # 3. Juntar Resultados preservando o alinhamento com y_test_curr
                            n_samples_eval = len(y_test_curr)
                            yp_clean = np.empty(n_samples_eval, dtype=np.int32)
                            yp_clean_prob = np.empty((n_samples_eval, num_classes), dtype=np.float32)
                            yp_adv_dict = {atk: np.empty(n_samples_eval, dtype=np.int32) for atk in active_adv_dict.keys()}
                            yp_adv_prob_dict = {atk: np.empty((n_samples_eval, num_classes), dtype=np.float32) for atk in active_adv_dict.keys()}
                            pid_adv_time = {atk: {} for atk in active_adv_dict.keys()}
                            
                            for r_worker in results_sorted:
                                start, end = r_worker['start'], r_worker['end']
                                pid = r_worker['pid']
                                yp_clean[start:end] = r_worker['yp_clean']
                                yp_clean_prob[start:end] = r_worker['yp_clean_prob']
                                for atk in active_adv_dict.keys():
                                    yp_adv_dict[atk][start:end] = r_worker['yp_adv'][atk]
                                    yp_adv_prob_dict[atk][start:end] = r_worker['yp_adv_prob'][atk]
                                    pid_adv_time[atk][pid] = pid_adv_time[atk].get(pid, 0.0) + r_worker['infer_time_adv'][atk]
                            
                            assert len(yp_clean) == len(y_test_curr)
                            for atk in active_adv_dict.keys():
                                assert len(yp_adv_dict[atk]) == len(y_test_curr)
                            
                            m_clean = calculate_miss_rates(y_test_curr, yp_clean, yp_clean_prob, "Clean", class_names_curr)
                            
                            # Como treinamos em paralelo, pegamos o tempo de treino do worker mais lento
                            train_time_real = max([r['train_time'] for r in results])
                            
                            for atk_name in active_adv_dict.keys():
                                yp_adv = np.array(yp_adv_dict[atk_name])
                                yp_adv_prob = np.array(yp_adv_prob_dict[atk_name])
                                infer_time_adv_real = max(pid_adv_time[atk_name].values()) if pid_adv_time[atk_name] else 0.0
                                
                                m_adv = calculate_miss_rates(y_test_curr, yp_adv, yp_adv_prob, "Adv", class_names_curr)
                                acc_drop = m_clean['Clean_Acc'] - m_adv['Adv_Acc']
                                
                                row = {
                                    'Dataset': dataset_name,
                                    'Attack': atk_name,
                                    'Resolution': res, 
                                    'AddressSize': addr, 
                                    'BloomSize': bloom,
                                    'Hash': h, 
                                    'TrainTime_s': train_time_real, 
                                    'InferTime_Adv_s': infer_time_adv_real,    
                                    'Acc_Drop_pp': acc_drop*100
                                }
                                row.update(m_clean)
                                row.update(m_adv)
                                
                                row_df = pd.DataFrame([row])
                                file_exists = os.path.exists(csv_name)
                                row_df.to_csv(csv_name, mode='a', header=not file_exists, index=False)
                                
                                file_tag = f"bloom_wisard_{dataset_name}_{mode}_{enc_type}_{atk_name}_R{res}_A{addr}_B{bloom}_H{h}"

                                save_curve_npz(
                                    CURVES_DIR / f"{file_tag}.npz",
                                    model_name=f"Bloom WiSARD ({enc_type})",
                                    attack_name=atk_name,
                                    y_true=y_test_curr,
                                    y_prob_clean=yp_clean_prob,
                                    y_prob_adv=yp_adv_prob,
                                    class_names=class_names_curr
                                )
                                
                            del parallel_out, results, results_sorted, yp_clean, yp_clean_prob, yp_adv_dict, yp_adv_prob_dict, yp_adv_prob, yp_adv
                            gc.collect()

            # <-- ALINHADO COM O FOR DA RESOLUÇÃO
            del X_train_bin, X_test_bin, bin_adv_dict, multi_adv_dict
            gc.collect()

    # <-- ALINHADO COM O FOR DO DATASET
    del attacks_dict_bin, attacks_dict_multi, X_train, X_test
    gc.collect()

# <-- ALINHADO À MARGEM ESQUERDA
print("\n🚀 EXPERIMENTO CONCLUÍDO COM SUCESSO!")



>>> A INICIAR EXPERIMENTOS: UNSW-NB15


C:\Users\root.REDE-LAGESED\AppData\Local\Temp\ipykernel_10544\2759695987.py:39: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_train.select_dtypes(include=['object']).columns


>>> A aplicar Scaling e One-Hot Encoding...
>>> A gerar Ataque C&W L2 via surrogate MLP...
   -> Treinando surrogate MLP para C&W (Binário)...


Gerando C&W Binário: 100%|██████████| 644/644 [20:24<00:00,  1.90s/lote]


   [C&W Binário] Sucesso no surrogate: 0.00% | L2 médio=0.0000 | L2 mediano=0.0000 | L2 máx=0.0000

   -> Treinando surrogate MLP para C&W (Multiclasse)...


...

   [C&W Multiclasse] Sucesso no surrogate: 0.00% | L2 médio=0.0000 | L2 mediano=0.0000 | L2 máx=0.0000

[UNSW-NB15 | GAUSSIAN | Res=10] Binarização C...
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [DISK] Memmap root: D:\wisard_memmap | livre≈9281.1 GB | estimado necessário≈80.8 GB
     [MEM] RAM disponível≈241.6 GB | RAM utilizável≈125.0 GB | pico estimado/worker≈2.0 GB | n_jobs seguro=23/23
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [DISK] Memmap root: D:\wisard_memmap | livre≈9281.1 GB | estimado necessário≈80.8 GB
     [MEM] RAM disponível≈241.6 GB | RAM utilizável≈125.0 GB | pico estimado/worker≈2.0 GB | n_jobs seguro=23/23
     [Addr=5 | Bloom=2048 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [DISK] Memmap root: D:\wisard_memmap | livre≈9281.1 GB | estimado necessário≈80.8 GB
     [MEM] RAM disponível≈241.6 GB | RAM u

Gerando C&W Binário: 100%|██████████| 3939/3939 [1:59:27<00:00,  1.82s/lote]


   [C&W Binário] Sucesso no surrogate: 0.00% | L2 médio=0.0000 | L2 mediano=0.0000 | L2 máx=0.0000
   -> Treinando surrogate MLP para C&W (Multiclasse)...


Gerando C&W Multiclasse:  53%|█████▎    | 2090/3939 [1:04:04<56:41,  1.84s/lote]  


KeyboardInterrupt: 